In [ ]:
import ee
import folium
import geemap
import geopandas as gpd
import json
import os

# Inicialize o Earth Engine
ee.Initialize()

date1 = input("digite a primeira data, as datas precisão ser 'Mês e Dia' exemplo: 01-01")
date2 = input("digite a segunda data")
print(f"período {date1} e {date2}")
def obtem_ano(ano):
    # Carrega o shapefile e filtra para o PARNA Serra da Canastra
    pnsc = gpd.read_file(r"G:\Meu Drive\@EquipeGEO\zz.Bases\ICMBio\UC_Fed_nov_2020.shp")
    pnsc = pnsc.loc[pnsc["nome"] == "PARQUE NACIONAL DA SERRA DA CANASTRA"]
    pnsc_geojson = pnsc.to_json()

    # Converte o shapefile para um objeto Earth Engine
    pnsc_ee = ee.FeatureCollection(json.loads(pnsc_geojson))

    # Função para selecionar a coleção Landsat com base no ano
    def selecionarColecaoLandsat(ano):
        if ano >= 1984 and ano <= 1999:
            return "LANDSAT/LT05/C02/T1_L2"  # Landsat 5
        elif ano >= 1999 and ano <= 2012:
            return "LANDSAT/LE07/C02/T1_L2"  # Landsat 7
        elif ano >= 2013:
            return "LANDSAT/LC08/C02/T1_L2"  # Landsat 8
        else:
            raise ValueError("Ano fora do intervalo disponível para coleções Landsat")

    # Função para mascarar nuvens e sombras
    def maskLandsat(image):
        qa = image.select('QA_PIXEL')
        cloud = qa.bitwiseAnd(1 << 3).eq(0)
        shadow = qa.bitwiseAnd(1 << 5).eq(0)
        mask = cloud.And(shadow)
        return image.updateMask(mask)

    # Parâmetros de entrada
    data_inicio = f"{ano}-{date1}"
    data_fim = f"{ano}-{date2}"

    # Seleciona a coleção com base no ano
    colecao = selecionarColecaoLandsat(ano)

    # Filtra e aplica a máscara na coleção de imagens
    dataset = ee.ImageCollection(colecao) \
        .filterDate(data_inicio, data_fim) \
        .filterBounds(pnsc_ee) \
        .map(maskLandsat)

    # Seleciona as bandas para a visualização True Color e NBR
    if ano <= 2012:  # Landsat 5 e 7
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B4", "SR_B7"]).rename("NBR"))
    else:  # Landsat 8
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B5", "SR_B7"]).rename("NBR"))

    # Obtém o NBR mínimo (severidade máxima)
    nbrMin = nbr.min()

    # Calcula o centróide do PNSC
    centroid = pnsc_ee.geometry().centroid()

    # Inicializa o mapa centrado no centróide do PNSC
    Map = geemap.Map(center=(centroid.coordinates().get(1).getInfo(), centroid.coordinates().get(0).getInfo()), zoom=10)

    # Adiciona camadas ao mapa
    Map.addLayer(nbrMin.updateMask(nbrMin.lte(0)), 
             {"min": -1, "max": 0, "palette": ["orange"]}, 
             "NBR (Valores entre -1 e 0)")
    Map.addLayer(ee.Image().paint(pnsc_ee, 0, 2), {}, "Limite do PARNA Serra da Canastra")

    
    return Map


mapa = obtem_ano(2015)
mapa


período 07-01 e 10-31


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [15]:
import ee
import folium
import geemap
import geopandas as gpd
import json
import os

# Inicialize o Earth Engine
ee.Initialize()

date1 = input("Digite a primeira data, as datas precisam ser 'Mês e Dia' exemplo: 01-01: ")
date2 = input("Digite a segunda data: ")
print(f"Período: {date1} e {date2}")

def obtem_ano(ano):
    # Carrega o shapefile e filtra para o PARNA Serra da Canastra
    pnsc = gpd.read_file(r"G:\Meu Drive\@EquipeGEO\zz.Bases\ICMBio\UC_Fed_nov_2020.shp")
    pnsc = pnsc.loc[pnsc["nome"] == "PARQUE NACIONAL DA SERRA DA CANASTRA"]
    pnsc_geojson = pnsc.to_json()

    # Converte o shapefile para um objeto Earth Engine
    pnsc_ee = ee.FeatureCollection(json.loads(pnsc_geojson))

    # Função para selecionar a coleção Landsat com base no ano
    def selecionarColecaoLandsat(ano):
        if 1984 <= ano <= 1999:
            return "LANDSAT/LT05/C02/T1_L2"  # Landsat 5
        elif 1999 <= ano <= 2012:
            return "LANDSAT/LE07/C02/T1_L2"  # Landsat 7
        elif ano >= 2013:
            return "LANDSAT/LC08/C02/T1_L2"  # Landsat 8
        else:
            raise ValueError("Ano fora do intervalo disponível para coleções Landsat")

    # Função para mascarar nuvens e sombras
    def maskLandsat(image):
        qa = image.select('QA_PIXEL')
        cloud = qa.bitwiseAnd(1 << 3).eq(0)
        shadow = qa.bitwiseAnd(1 << 5).eq(0)
        mask = cloud.And(shadow)
        return image.updateMask(mask)

    # Função para mascarar água usando NDWI
    def maskWater(image):
        ndwi = image.normalizedDifference(["SR_B3", "SR_B5"])  # B3 = Verde, B5 = Infravermelho Próximo (Landsat 8)
        water_mask = ndwi.lt(0)  # Mantém apenas áreas onde NDWI < 0 (não água)
        return image.updateMask(water_mask)

    # Parâmetros de entrada
    data_inicio = f"{ano}-{date1}"
    data_fim = f"{ano}-{date2}"

    # Seleciona a coleção com base no ano
    colecao = selecionarColecaoLandsat(ano)

    # Filtra e aplica a máscara na coleção de imagens
    dataset = ee.ImageCollection(colecao) \
        .filterDate(data_inicio, data_fim) \
        .filterBounds(pnsc_ee) \
        .map(maskLandsat) \
        .map(maskWater)  # Aplica a máscara de água

    # Seleciona as bandas para a visualização True Color e NBR
    if ano <= 2012:  # Landsat 5 e 7
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B4", "SR_B7"]).rename("NBR"))
    else:  # Landsat 8
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B5", "SR_B7"]).rename("NBR"))

    # Obtém o NBR mínimo (severidade máxima)
    nbrMin = nbr.min()

    # Calcula o centróide do PNSC
    centroid = pnsc_ee.geometry().centroid()

    # Inicializa o mapa centrado no centróide do PNSC
    Map = geemap.Map(center=(centroid.coordinates().get(1).getInfo(), centroid.coordinates().get(0).getInfo()), zoom=10)

    # Adiciona camadas ao mapa
    Map.addLayer(nbrMin.updateMask(nbrMin.lte(0)), 
                 {"min": -1, "max": 0, "palette": ["orange"]}, 
                 "NBR (Valores entre -1 e 0)")
    Map.addLayer(ee.Image().paint(pnsc_ee, 0, 2), {}, "Limite do PARNA Serra da Canastra")

    return Map

# Gera o mapa para o ano 2015
mapa = obtem_ano(2015)

# Salva o mapa em um arquivo HTML
mapa.save("mapa_serra_canastra.html")

print("Mapa salvo com sucesso como 'mapa_serra_canastra.html'")
mapa

Período: 07-01 e 10-31
Mapa salvo com sucesso como 'mapa_serra_canastra.html'


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [1]:
import ee
import geemap
import geopandas as gpd
import json
import requests
import zipfile
import os
from io import BytesIO
from shapely.geometry import Point

def baixar_e_extrair_zip(url_base, ano):
    nome_arquivo_zip = f"focos_br_ref_{ano}.zip"
    url = f"{url_base}/{nome_arquivo_zip}"

    # Faz o download do arquivo ZIP
    response = requests.get(url)
    if response.status_code != 200:
        raise FileNotFoundError(f"Erro ao baixar o arquivo ZIP: {url}")

    with open(nome_arquivo_zip, 'wb') as f:
        f.write(response.content)

    if not os.path.exists(nome_arquivo_zip):
        raise FileNotFoundError(f"Arquivo ZIP não foi baixado: {nome_arquivo_zip}")

    # Extrai o conteúdo do ZIP
    with zipfile.ZipFile(nome_arquivo_zip, 'r') as zip_ref:
        arquivos_no_zip = zip_ref.namelist()
        print("Arquivos no ZIP:", arquivos_no_zip)

        # Procura pelo arquivo CSV dentro do ZIP
        arquivo_csv = next((arquivo for arquivo in arquivos_no_zip 
                            if arquivo.endswith('.csv')), None)

        if not arquivo_csv:
            raise FileNotFoundError(f"Nenhum arquivo CSV encontrado no ZIP: {nome_arquivo_zip}")

        zip_ref.extract(arquivo_csv)

    # Verifica ou cria o diretório "shp"
    pasta_shp = os.path.join(os.getcwd(), 'shp')
    if not os.path.exists(pasta_shp):
        os.makedirs(pasta_shp)

    # Move o arquivo CSV extraído para a pasta "shp"
    caminho_csv_origem = os.path.join(os.getcwd(), arquivo_csv)
    caminho_csv_destino = os.path.join(pasta_shp, arquivo_csv)

    os.rename(caminho_csv_origem, caminho_csv_destino)

    print(f"Arquivo CSV movido para a pasta 'shp': {caminho_csv_destino}")
    return caminho_csv_destino

def ler_csv_com_geopandas(caminho_csv):
    print(f'Lendo {caminho_csv} com GeoPandas...')
    return gpd.read_file(caminho_csv)

ee.Initialize()

def obtem_ano(ano):
    #date1 = input("Digite a primeira data: ")
    #date2 = input("Digite a segunda data: ")
    #print(f"Período: {date1} e {date2}")

    data_inicio = f"{ano}-07-01"
    data_fim = f"{ano}-10-31"
    
    pnsc = gpd.read_file(r"G:\Meu Drive\@EquipeGEO\zz.Bases\ICMBio\UC_Fed_nov_2020.shp")
    pnsc = pnsc.loc[pnsc["nome"] == "PARQUE NACIONAL DA SERRA DA CANASTRA"]
    pnsc_geojson = pnsc.to_json()

    pnsc_ee = ee.FeatureCollection(json.loads(pnsc_geojson))

    url_base = 'https://dataserver-coids.inpe.br/queimadas/queimadas/focos/csv/anual/Brasil_sat_ref/'
    arquivo_csv = baixar_e_extrair_zip(url_base, ano)
    gdf = ler_csv_com_geopandas(arquivo_csv)
    gdf['geometry'] = gpd.points_from_xy(gdf['lon'], gdf['lat'])
    gdf = gpd.GeoDataFrame(gdf, geometry='geometry')

    # Verifique e converta colunas do tipo bytes para strings
    for col in gdf.columns:
        if gdf[col].dtype == object:  # Apenas verifica colunas do tipo 'object'
            gdf[col] = gdf[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

    focos = gdf.loc[(gdf['estado'] == 'MINAS GERAIS') & (gdf['data_pas'] >= data_inicio) & (gdf['data_pas'] <= data_fim)]
    focos_geojson = focos.to_json()

    focos_ee = ee.FeatureCollection(json.loads(focos_geojson))

    def selecionarColecaoLandsat(ano):
        if 1984 <= ano <= 1999:
            return "LANDSAT/LT05/C02/T1_L2"
        elif 1999 <= ano <= 2012:
            return "LANDSAT/LE07/C02/T1_L2"
        elif ano >= 2013:
            return "LANDSAT/LC08/C02/T1_L2"
        else:
            raise ValueError("Ano fora do intervalo disponível para coleções Landsat")

    # Função para mascarar nuvens e sombras
    def maskLandsat(image):
        qa = image.select('QA_PIXEL')
        cloud = qa.bitwiseAnd(1 << 3).eq(0)
        shadow = qa.bitwiseAnd(1 << 5).eq(0)
        mask = cloud.And(shadow)
        return image.updateMask(mask)

    # Função para mascarar água usando NDWI
    def maskWater(image):
        ndwi = image.normalizedDifference(["SR_B3", "SR_B5"])  # B3 = Verde, B5 = Infravermelho Próximo (Landsat 8)
        water_mask = ndwi.lt(0)  # Mantém apenas áreas onde NDWI < 0 (não água)
        return image.updateMask(water_mask)

    colecao = selecionarColecaoLandsat(ano)

    # Filtra e aplica a máscara na coleção de imagens
    dataset = ee.ImageCollection(colecao) \
        .filterDate(data_inicio, data_fim) \
        .filterBounds(pnsc_ee) \
        .map(maskLandsat) \
        .map(maskWater)  # Aplica a máscara de água

    if ano <= 2012:
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B4", "SR_B7"]).rename("NBR"))
    else:
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B5", "SR_B7"]).rename("NBR"))

    nbrMin = nbr.min()

    centroid = pnsc_ee.geometry().centroid()

    Map = geemap.Map(center=(centroid.coordinates().get(1).getInfo(), centroid.coordinates().get(0).getInfo()), zoom=10)

    Map.addLayer(nbrMin.updateMask(nbrMin.lte(0)), 
                 {"min": -1, "max": 0, "palette": ["orange"]}, 
                 "NBR (Valores entre -1 e 0)")
    Map.addLayer(ee.Image().paint(pnsc_ee, 0, 2), {}, "Limite do PARNA Serra da Canastra")

    Map.addLayer(focos_ee, {"color": "red"}, "Focos de Calor")

    return Map

In [43]:
# Testando para o ano de 2022
mapa = obtem_ano(2022)
mapa

Período: 07-01 e 10-31
Arquivos no ZIP: ['focos_br_ref_2022.csv']
Lendo focos_br_ref_2022.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [44]:
# Testando para o ano de 2021
mapa = obtem_ano(2021)
mapa

Período: 07-01 e 10-31
Arquivos no ZIP: ['focos_br_ref_2021.csv']
Lendo focos_br_ref_2021.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [46]:
# Testando para o ano de 2020
mapa = obtem_ano(2020)
mapa

Arquivos no ZIP: ['focos_br_ref_2020.csv']
Lendo focos_br_ref_2020.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [47]:
# Testando para o ano de 2019
mapa = obtem_ano(2019)
mapa

Arquivos no ZIP: ['focos_br_ref_2019.csv']
Lendo focos_br_ref_2019.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [48]:
# Testando para o ano de 2018
mapa = obtem_ano(2018)
mapa

Arquivos no ZIP: ['focos_br_ref_2018.csv']
Lendo focos_br_ref_2018.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [49]:
# Testando para o ano de 2017
mapa = obtem_ano(2017)
mapa

Arquivos no ZIP: ['focos_br_ref_2017.csv']
Lendo focos_br_ref_2017.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [50]:
# Testando para o ano de 2016
mapa = obtem_ano(2016)
mapa

Arquivos no ZIP: ['focos_br_ref_2016.csv']
Lendo focos_br_ref_2016.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [51]:
# Testando para o ano de 2015
mapa = obtem_ano(2015)
mapa

Arquivos no ZIP: ['focos_br_ref_2015.csv']
Lendo focos_br_ref_2015.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [52]:
# Testando para o ano de 2014
mapa = obtem_ano(2014)
mapa

Arquivos no ZIP: ['focos_br_ref_2014.csv']
Lendo focos_br_ref_2014.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [53]:
# Testando para o ano de 2013
mapa = obtem_ano(2013)
mapa

Arquivos no ZIP: ['focos_br_ref_2013.csv']
Lendo focos_br_ref_2013.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [54]:
# Testando para o ano de 2012
mapa = obtem_ano(2012)
mapa

Arquivos no ZIP: ['focos_br_ref_2012.csv']
Lendo focos_br_ref_2012.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [55]:
# Testando para o ano de 2011
mapa = obtem_ano(2011)
mapa

Arquivos no ZIP: ['focos_br_ref_2011.csv']
Lendo focos_br_ref_2011.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [56]:
# Testando para o ano de 2010
mapa = obtem_ano(2010)
mapa

Arquivos no ZIP: ['focos_br_ref_2010.csv']
Lendo focos_br_ref_2010.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [58]:
# Testando para o ano de 2009
mapa = obtem_ano(2009)
mapa

Arquivos no ZIP: ['focos_br_ref_2009.csv']
Arquivo CSV movido para a pasta 'shp': c:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_br_ref_2009.csv
Lendo c:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_br_ref_2009.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [59]:
# Testando para o ano de 2008
mapa = obtem_ano(2008)
mapa

Arquivos no ZIP: ['focos_br_ref_2008.csv']
Arquivo CSV movido para a pasta 'shp': c:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_br_ref_2008.csv
Lendo c:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_br_ref_2008.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [ ]:
# Testando para o ano de 2007
mapa = obtem_ano(2007)
mapa

In [ ]:
# Testando para o ano de 2006
mapa = obtem_ano(2006)
mapa

In [ ]:
# Testando para o ano de 2005
mapa = obtem_ano(2005)
mapa

In [ ]:
# Testando para o ano de 2004
mapa = obtem_ano(2004)
mapa

In [ ]:
# Testando para o ano de 2003
mapa = obtem_ano(2003)
mapa